In [1]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

from caveat.label_encoding import TokenAttributeEncoder
from caveat.mine_yz import DataModule, MutualInformationEstimator, YZDataset
from caveat.models.continuous.cvae_lstm import LabelEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

Device: cuda


In [2]:
def latest(path: Path):
    versions = sorted(
        [
            d
            for d in path.iterdir()
            if d.is_dir() and d.name.startswith("version")
        ]
    )
    return Path(versions[-1])


def iter_models(path: Path):
    for dir in path.iterdir():
        if dir.is_dir():
            yield latest(dir)

In [3]:
label_encoder = TokenAttributeEncoder(
    config={
        "gender": "nominal",
        "age_group": "nominal",
        "car_access": "nominal",
        "work_status": "nominal",
        "income": "nominal",
    }
)


def custom_loader(
    root: Path, label_encoder, random_z: bool = False, embed_z: bool = False
):
    for path in iter_models(root):
        ys = pd.read_csv(path / "test_inference" / "input_attributes.csv")
        ys, _ = label_encoder.encode(ys)
        zs = pd.read_csv(path / "test_inference" / "zs.csv", header=None).values
        if random_z:
            rng = np.random.default_rng()
            zs = rng.normal(loc=0.0, scale=1.0, size=zs.shape)
        if embed_z:
            # strong MI example
            embedder = LabelEncoder(
                label_embed_sizes=label_encoder.label_kwargs[
                    "label_embed_sizes"
                ],
                hidden_size=zs.shape[1],
            )
            zs = embedder(ys).detach().numpy()
        yield ys, zs

Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            


In [4]:
class MinerNet(nn.Module):
    def __init__(
        self,
        label_embed_sizes,
        hidden_size=512,
        latent_dim=6,
        block_depth=4,
        dropout=0.3,
    ):
        super(MinerNet, self).__init__()

        self.label_embed = LabelEncoder(
            label_embed_sizes=label_embed_sizes, hidden_size=hidden_size
        )
        self.z_embed = nn.Sequential(
            nn.Linear(in_features=latent_dim, out_features=hidden_size),
            nn.LeakyReLU(),
        )

        blocks = []
        for _ in range(block_depth - 1):
            blocks.append(nn.Linear(hidden_size, hidden_size))
            if dropout > 0:
                blocks.append(nn.Dropout(dropout))
            blocks.append(nn.LeakyReLU())
        self.blocks = nn.Sequential(*blocks, nn.Linear(hidden_size, 1))

    def forward(self, ys, zs):
        h1 = self.label_embed(ys.long())
        h2 = self.z_embed(zs)
        return self.blocks(h1 + h2)

In [5]:
data_loaders = {
    "random": custom_loader(
        Path("../logs/TRB/cvae"), label_encoder, random_z=True
    ),
    "cvae": custom_loader(Path("../logs/TRB/cvae"), label_encoder),
    "vae": custom_loader(Path("../logs/TRB/vae_labels"), label_encoder),
    "strong": custom_loader(
        Path("../logs/TRB/vae_labels"), label_encoder, embed_z=True
    ),
}
results = {}
for name, loader in data_loaders.items():
    model_results = []
    for i, (ys, zs) in enumerate(loader):

        logger = TensorBoardLogger("logs/zy", name=f"{name}_{i}")
        dataset = YZDataset(ys=ys, zs=zs)
        loader = DataModule(
            dataset=dataset,
            val_split=0.1,
            test_split=0.1,
            batch_size=1024,
            num_workers=8,
            pin_memory=False,
        )

        net = MinerNet(
            label_embed_sizes=label_encoder.label_kwargs["label_embed_sizes"],
            hidden_size=512,
            block_depth=6,
            latent_dim=6,
            dropout=0.3,
        )

        kwargs = {"alpha": 1, "lr": 1e-3, "weight_decay": 1e-3}
        model = MutualInformationEstimator(net=net, **kwargs)
        trainer = Trainer(
            min_epochs=10,
            max_epochs=500,
            accelerator=device,
            devices=1,
            enable_progress_bar=False,
            logger=logger,
            enable_checkpointing=True,
            callbacks=[
                EarlyStopping(monitor="val_loss", patience=20),
                ModelCheckpoint(
                    monitor="val_loss", save_top_k=2, save_weights_only=False
                ),
            ],
        )
        trainer.fit(model, datamodule=loader)
        mi = trainer.test(ckpt_path="best", datamodule=loader)[0]["test_mi"]
        model_results.append(mi)
    results[name] = {
        "mean": np.mean(model_results),
        "var": np.var(model_results),
    }

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │  1.8217600882053375e-05   │
│          test_mi          │  -1.8217600882053375e-05  │
└───────────────────────────┴───────────────────────────┘

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │  -0.00019101053476333618  │
│          test_mi          │  0.00019101053476333618   │
└───────────────────────────┴───────────────────────────┘

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    -2.4009594917297363    │
│          test_mi          │    2.4009594917297363     │
└───────────────────────────┴───────────────────────────┘

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    -3.9392707347869873    │
│          test_mi          │    3.9392707347869873     │
└───────────────────────────┴───────────────────────────┘

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    -4.822325229644775     │
│          test_mi          │     4.822325229644775     │
└───────────────────────────┴───────────────────────────┘

In [6]:
for name, result in results.items():
    print(f"\tResults for {name}: {result}")

	Results for random: {'mean': 6.536468863487244e-05, 'var': 5.342797632576502e-09}
	Results for cvae: {'mean': 0.17016934156417846, 'var': 0.008924022788872266}
	Results for vae: {'mean': 0.3406977355480194, 'var': 0.0015793720033414616}
	Results for strong: {'mean': 3.9802443027496337, 'var': 0.8028825184150626}


In [10]:
df = pd.DataFrame.from_dict(results, orient="index")
print(df.to_latex(float_format="{:.4f}".format))

\begin{tabular}{lrr}
\toprule
 & mean & var \\
\midrule
random & 0.0001 & 0.0000 \\
cvae & 0.1702 & 0.0089 \\
vae & 0.3407 & 0.0016 \\
strong & 3.9802 & 0.8029 \\
\bottomrule
\end{tabular}

